# 02 — How did the agent reach 2022-03-09?

We audit one real Luna review of synthetic case `SYNX03`.

- **Task:** find the earliest diagnosis date.
- **Agent answer:** `20220309`.
- **Synthetic gold:** `20220309`.
- **First conclusion:** the final answer is correct.

A correct answer can still come from a bad process. We therefore ask: **what did the
agent look at, what did it judge, and which step is still risky?**

Think of the raw trace as a long receipt. It proves what happened. The Decision Chain
groups that receipt into a short set of questions a human can judge.


**Checked reading copy.** The saved outputs below come from completed real-provider runs. Implementation cells are omitted here for readability; open [`02_trace_to_decision_chain.ipynb`](./02_trace_to_decision_chain.ipynb) to inspect or rerun the code.


## 1. What does the raw trace look like?

The full trace has 22 observable events. Reading every JSON field is possible, but it
is not how a reviewer should start. First group the receipt by purpose.


| Stage | Raw trace says |
|---|---|
| Inventory | Two pages: 200 + 112 = 312 note headers |
| Search | Keywords: diagnos, cancer, malignan, carcinoma |
| Candidates | 3 unique notes surfaced |
| Read | Opened 3 notes |
| Judge | Recorded 3 evidence judgments |
| Proof | Saved 3 exact evidence spans |
| Submit | FOUND → 20220309 |

### One actual event (trimmed only for display)

```json
{
  "seq": 14,
  "kind": "tool_call",
  "tool": "record_finding",
  "args": {
    "note_id": "Surgical-Pathology-Report_2022-03-09",
    "standing": "can_establish",
    "because": "A pathology interpretation with a final diagnosis of squamous cell carcinoma is a qualifying witness and establishes the diagnosis on its service date."
  },
  "result": {
    "ok": true,
    "testimony_ref": "decision:14",
    "server_sealed_receipt": true
  }
}
```

A raw event is excellent evidence: it tells us which note, which tool, which choice,
and whether the server sealed the record. But it does not tell the reviewer where one
important judgment ends and the next begins.

## 2. Turn the receipt into eight questions

Luna reconstructs fixed ReAct cycles into **Decision Episodes**. One episode means one
consequential question that one human can mark right, wrong, or uncertain.

```text
312-note inventory
        ↓
4 keyword searches
        ↓
3 candidate notes
        ↓
2/14 ❌ suspicious   3/9 ✅ biopsy   3/11 ✅ physician diagnosis
        ↓
earliest qualifying date = 3/9
        ↓
submit 20220309
```


| # | Question for the reviewer | Agent's choice | Human audit reading + source |
|---|---|---|---|
| 1 | Was the chart inventory complete? | Inventory all 312 note headers | ✓ Complete after page 2 · Agent said this at runtime |
| 2 | Which keywords should we use to search for diagnosis evidence? | Search diagnosis, cancer, malignancy, carcinoma | ⚠ Main audit point: are four terms enough? · Agent said this at runtime |
| 3 | Which surfaced notes should be opened? | Open the 3 surfaced candidate notes | ⚠ Notes opened, but selection reason was not recorded · Action observed; reason reconstructed by Luna |
| 4 | Can the 2/14 suspicious cytology establish diagnosis? | No — suspicious only | ✓ Correctly rejects ambiguous cytology alone · Agent said this at runtime |
| 5 | Can the 3/9 definitive biopsy establish diagnosis? | Yes — qualifying evidence | ✓ Definitive pathology qualifies · Agent said this at runtime |
| 6 | Can the 3/11 physician diagnosis establish diagnosis? | Yes — qualifying evidence | ✓ Physician diagnosis qualifies, but is later · Agent said this at runtime |
| 7 | Which of the three dates is the earliest qualifying date? | Choose 2022-03-09 | ✓ Applies the earliest-qualifying rule correctly · Agent said this at runtime |
| 8 | Is there enough evidence to stop and submit? | Stop and submit 20220309 | △ Correct, but inherits the search-coverage risk · Agent said this at runtime |

## 3. Follow the clinical decision

The three evidence judgments are the heart of the answer:

1. `2022-02-14`: atypical cells were only **suspicious**; biopsy was recommended. This
   note alone does not establish the date.
2. `2022-03-09`: the biopsy's final diagnosis is squamous cell carcinoma. This does
   establish the date.
3. `2022-03-11`: the physician says the mass clinically represents malignancy. This
   also qualifies, but it is later.
4. The task asks for the **earliest qualifying** date, so `2022-03-09` wins.

This is the same explanation a careful human reviewer would ask a colleague to give.


## 4. Where should the human spend time?

Not on the final date comparison. The weak point is earlier: **the agent chose four
search terms.** Those calls really happened, and they found the three decisive notes.
But the policy says what evidence counts; it does not prove that these four terms cover
every possible clinical wording.


- **Agent's question:** The inventory shows oncology and physician progress notes beginning 2012-2013, plus surgical pathology in 2022; the first diagnosis may be documented in an earlier physician note or pathology record.
- **Agent's choice:** Search the chart using the keyword batch diagnosis, cancer, malignancy, and carcinoma to locate candidate diagnosis statements and retrospective dates.
- **Agent's stated reason:** These terms directly target physician diagnostic impressions and pathology/oncology references; the inventory decision identifies the relevant document families and time span.
- **Raw proof:** events 5, 6, 7, 8 executed `diagnos, cancer, malignan, carcinoma` and surfaced 3 unique notes.
- **Reviewer question:** would a different wording escape all four searches?

**If the answer is yes—or we cannot rule it out—improve the retrieval guideline here.** Changing the later date-conflict rule would not fix a missed note.

## 5. Is the Decision level better to read?

Yes—for choosing **where to audit**. No—as a replacement for execution evidence.


```text
158 Codex protocol records (harness detail)
→ 22 observable Langtrace events
→ 21 fixed ReAct cycles
→ 8 human-auditable decisions
```

- 7/8 choices were explicitly stated by the agent at runtime.
- 1/8 choice was recovered from observed actions; its explanation is visibly labeled as Luna reconstruction.
- 8/8 decisions link back to raw events.
- 21/21 cycles are accounted for exactly once.

**Bottom line:** use the Decision Chain to find the questionable step. Then use the raw trace and provenance to verify what actually happened.